# AI detection tools evaluation
## Dataset download

In [1]:
from datasets import load_dataset
ds = load_dataset("silentone0725/ai-human-text-detection-v1")

Repo card metadata block was not found. Setting CardData to empty.


In [2]:
ds.save_to_disk("ai_human_text_detection_data")

Saving the dataset (0/1 shards):   0%|          | 0/36744 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7874 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7874 [00:00<?, ? examples/s]

In [3]:
data = ds["train"]

TEXT_COL = "text"
LABEL_COL = "label"

ai_texts = []
human_texts = []

for item in data:

    text = item[TEXT_COL].strip()
    if not text:
        continue
    text = text.strip()
    label = item[LABEL_COL]

    if len(text.split()) < 50:
        continue

    if label == "ai" and len(ai_texts) < 25:
        ai_texts.append(text)
    elif label == "human" and len(human_texts) < 25:
        human_texts.append(text)

    if len(ai_texts) == 25 and len(human_texts) == 25:
        break

def save_to_file(texts, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for i, text in enumerate(texts, start=1):
            f.write(f"{i}\n{text}\n\n")

save_to_file(ai_texts, "tools_test_data/ai_texts.txt")
save_to_file(human_texts, "tools_test_data/human_texts.txt")

## Models inference

In [44]:
from datasets import load_dataset
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
import numpy as np

In [ ]:
data = ds["train"].shuffle(seed=42).select(range(200))

TEXT_COL = "text"
LABEL_COL = "label"

detector_base = pipeline(
    "text-classification",
    model="roberta-base-openai-detector",
    truncation=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
y_true_roberta = []
y_pred_base_roberta = []

def map_label(pred_label):
    return 1 if pred_label == "Fake" else 0

for item in data:
    text = item[TEXT_COL]
    true_label = item[LABEL_COL]

    pred = detector_base(text)[0]
    pred_label = map_label(pred["label"])

    y_true_roberta.append(true_label)
    y_pred_base_roberta.append(pred_label)

In [ ]:
tokenizer_gpt2 = GPT2Tokenizer.from_pretrained("gpt2")
model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
model_gpt2.eval()

def compute_perplexity(text):
    encodings = tokenizer_gpt2(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = encodings.input_ids

    with torch.no_grad():
        outputs = model_gpt2(input_ids, labels=input_ids)
        loss = outputs.loss

    return torch.exp(loss).item()

def perplexity_to_label(ppl, threshold=50):
    return 1 if ppl < threshold else 0

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [54]:
y_pred_ppl = []

for item in data:
    text = item["text"]
    true_label = item["label"]

    ppl = compute_perplexity(text)
    pred_ppl = perplexity_to_label(ppl)

    y_pred_ppl.append(pred_ppl)

## Metrics
### RoBERTa

In [49]:
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label="ai")
    cm = confusion_matrix(y_true, y_pred, labels=["human", "ai"])

    print(f"{name}")
    print(f"Accuracy: {acc:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1: {f1:.3f}")
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print(f"False Positive Rate (FPR): {fpr:.3f}")
    print(f"False Negative Rate (FNR): {fnr:.3f}")

In [50]:
evaluate("RoBERTa Base", y_true_roberta, list(map(lambda x: 'human' if x == 0 else 'ai', y_pred_base_roberta)))

RoBERTa Base
Accuracy: 0.935
Precision: 0.963
Recall: 0.921
F1: 0.942
Confusion Matrix:
[[ 82   4]
 [  9 105]]
False Positive Rate (FPR): 0.047
False Negative Rate (FNR): 0.079


### GPT2

In [56]:
evaluate("GPT2", y_true_roberta, list(map(lambda x: 'human' if x == 0 else 'ai', y_pred_ppl)))

GPT2
Accuracy: 0.785
Precision: 0.726
Recall: 1.000
F1: 0.841
Confusion Matrix:
[[ 43  43]
 [  0 114]]
False Positive Rate (FPR): 0.500
False Negative Rate (FNR): 0.000


### GPTZero

In [57]:
import re
import pandas as pd

def parse_file(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    entries = re.split(r'\n(?=\d+[.,])', content)

    rows = []

    for entry in entries[1:]:
        entry = entry.strip()
        if not entry:
            continue

        try:
            first_comma = entry.find(',')
            second_comma = entry.find(',', first_comma + 1)

            id_ = entry[:first_comma].strip()
            text_and_rest = entry[first_comma+1:]

            last_comma = text_and_rest.rfind(',')
            second_last_comma = text_and_rest.rfind(',', 0, last_comma)

            text = text_and_rest[:second_last_comma].strip().strip('"')
            label = text_and_rest[second_last_comma+1:last_comma].strip()
            score = text_and_rest[last_comma+1:].strip()

            rows.append({
                "id": int(id_),
                "text": text,
                "true_label": label,
                "score": float(score)
            })

        except Exception as e:
            print(f"Skipping malformed entry:\n{entry[:100]}...\nError: {e}\n")

    return pd.DataFrame(rows)


df = parse_file("results/gprzero.csv")

print(df.head())
print(f"Loaded rows: {len(df)}")

   id                                               text true_label  score
0   1  We don't actually know what color dinosaurs li...         ai   0.00
1   2  A ship of the line was a type of warship that ...         ai   1.00
2   3  There are many complex factors that have contr...         ai   0.38
3   4  Elijah McCoy (1844-1929) was an African Americ...         ai   0.81
4   5  It's true that China is a major player in the ...         ai   0.05
Loaded rows: 30


In [ ]:
df["pred_label"] = df["score"].apply(lambda x: "ai" if x >= 0.5 else "human")

y_true_gpt = df["true_label"]
y_pred_gpt = df["pred_label"]

evaluate("GPTZero", y_true_gpt, y_pred_gpt)

GPTZero
Accuracy: 0.833
Precision: 1.000
Recall: 0.667
F1: 0.800
Confusion Matrix:
[[15  0]
 [ 5 10]]
False Positive Rate (FPR): 0.000
False Negative Rate (FNR): 0.333


### GPTZero with own and self generated text